# Baseline Modeling and Error Analysis

The objective of this notebook is to:
- Build baseline models,
- Compare their results,
- Analyze false negatives,
- Analyze false positives,
- Formulate hypotheses for further experiments.

## Environment and imports

Import libraries and evaluation metrics.

In [1]:
import pandas as pd
import numpy as np

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

## Helper functions

Define model evaluation and error-analysis helpers.

In [2]:
DECISION_THRESHOLD = 0.5

def evaluate_model(name, model, X_test, y_test, threshold=DECISION_THRESHOLD):
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba > threshold).astype(int)
    
    print(name)

    print("Decision Threshold:", threshold)
    print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
    print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 4))
    print("Recall:", round(recall_score(y_test, y_pred), 4))
    print("F1:", round(f1_score(y_test, y_pred), 4))
    print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))
    print("PR-AUC:", round(average_precision_score(y_test, y_proba), 4))

    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    
    print("Classification Report:\n", classification_report(y_test, y_pred, zero_division=0))


def get_metrics(name, model, X_test, y_test, threshold=DECISION_THRESHOLD):
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba > threshold).astype(int)

    return {
        "model": name,
        "decision_threshold": threshold,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba)
    }

def get_false_negatives(model):
    return debug_df[(debug_df['y_true']==1) & (debug_df[f'{model}_pred']== 0)]

## Load prepared data

Load scaled features and original-index metadata.

In [3]:
# Dane skalowane — do trenowania modeli
X_train = pd.read_csv(
    "../data/processed/X_train_scaled.csv",
    index_col="record_index"
)

X_test = pd.read_csv(
    "../data/processed/X_test_scaled.csv",
    index_col="record_index"
)

# Dane nieskalowane — do interpretacji rekordów i błędów modeli
X_train_raw = pd.read_csv(
    "../data/processed/X_train_raw.csv",
    index_col="record_index"
)

X_test_raw = pd.read_csv(
    "../data/processed/X_test_raw.csv",
    index_col="record_index"
)

# Target
y_train = pd.read_csv(
    "../data/processed/y_train.csv",
    index_col="record_index"
).squeeze("columns")

y_test = pd.read_csv(
    "../data/processed/y_test.csv",
    index_col="record_index"
).squeeze("columns")

### Validate loaded data

Check shapes, indexes and class distributions.

In [4]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

print(X_train.head())
print(y_train.value_counts())
print(y_test.value_counts())

(8000, 8)
(2000, 8)
(8000,)
(2000,)
              Type_L  Type_M  Process temperature [K]  Temperature difference  \
record_index                                                                    
3693               1       0                 0.874402               -0.800498   
590                1       0                -0.408876                1.900381   
6770               0       1                 0.536697               -0.500400   
1412               0       1                -0.071171                1.100121   
3298               1       0                 0.198993               -0.900531   

              Rotational speed [rpm]  Torque [Nm]  Power [W]  Tool wear [min]  
record_index                                                                   
3693                       -0.100148    -0.851496  -1.182391         1.485820  
590                        -0.122496    -0.329786  -0.421646         1.973739  
6770                       -0.167192     0.011332   0.052397        -1.30004

## Baseline models

Train simple reference and tree-based classifiers.

### Dummy classifier

Establish a majority-class reference score.

In [5]:
dummy = DummyClassifier(strategy='most_frequent')

dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)
y_proba_dummy = dummy.predict_proba(X_test)[:, 1]

In [6]:
evaluate_model('Dummy Classifier', dummy, X_test, y_test)

Dummy Classifier
Decision Threshold: 0.5
Accuracy: 0.967
Precision: 0.0
Recall: 0.0
F1: 0.0
ROC-AUC: 0.5
PR-AUC: 0.033
Confusion Matrix:
 [[1934    0]
 [  66    0]]
Classification Report:
               precision    recall  f1-score   support

           0       0.97      1.00      0.98      1934
           1       0.00      0.00      0.00        66

    accuracy                           0.97      2000
   macro avg       0.48      0.50      0.49      2000
weighted avg       0.94      0.97      0.95      2000



Observation: The dummy model has high accuracy but detects no failures. It is not useful for this task.



### Logistic regression

Train a linear baseline with balanced classes.

In [7]:
log_reg = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

log_reg.fit(X_train, y_train)

y_pred_log = log_reg.predict(X_test)
y_proba_log = log_reg.predict_proba(X_test)[:, 1]


In [8]:
evaluate_model('LogisticRegression', log_reg, X_test, y_test)

LogisticRegression
Decision Threshold: 0.5
Accuracy: 0.8475
Precision: 0.1595
Recall: 0.8485
F1: 0.2686
ROC-AUC: 0.9202
PR-AUC: 0.4192
Confusion Matrix:
 [[1639  295]
 [  10   56]]
Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.85      0.91      1934
           1       0.16      0.85      0.27        66

    accuracy                           0.85      2000
   macro avg       0.58      0.85      0.59      2000
weighted avg       0.97      0.85      0.89      2000



Observation: Logistic regression detects failures, but its low precision produces many false alarms.



### Random forest

Train a nonlinear ensemble baseline.

In [9]:
random_forest = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42
)

random_forest.fit(X_train, y_train)

evaluate_model('RandomForestClassifier', random_forest, X_test, y_test)

RandomForestClassifier
Decision Threshold: 0.5
Accuracy: 0.99
Precision: 0.8594
Recall: 0.8333
F1: 0.8462
ROC-AUC: 0.991
PR-AUC: 0.893
Confusion Matrix:
 [[1925    9]
 [  11   55]]
Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      0.99      1934
           1       0.86      0.83      0.85        66

    accuracy                           0.99      2000
   macro avg       0.93      0.91      0.92      2000
weighted avg       0.99      0.99      0.99      2000



Observation: Random Forest gives the best precision/recall balance, with 11 false negatives and 9 false positives.

### Gradient boosting

Train a sequential tree-based baseline.

In [10]:
gradient_boosting = GradientBoostingClassifier(
    random_state=42
)

gradient_boosting.fit(X_train, y_train)

evaluate_model('GradientBoostingClassifier', gradient_boosting, X_test, y_test)

GradientBoostingClassifier
Decision Threshold: 0.5
Accuracy: 0.9895
Precision: 0.8814
Recall: 0.7879
F1: 0.832
ROC-AUC: 0.9943
PR-AUC: 0.9145
Confusion Matrix:
 [[1927    7]
 [  14   52]]
Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      0.99      1934
           1       0.88      0.79      0.83        66

    accuracy                           0.99      2000
   macro avg       0.94      0.89      0.91      2000
weighted avg       0.99      0.99      0.99      2000



Observation: Gradient Boosting reduces false positives to 7, but misses 14 failures.

### Compare baseline models

Collect and compare classification metrics.

In [11]:
results = []

results.append(get_metrics("Dummy", dummy, X_test, y_test))
results.append(get_metrics("Logistic Regression", log_reg, X_test, y_test))
results.append(get_metrics("Random Forest", random_forest, X_test, y_test))
results.append(get_metrics("Gradient Boosting", gradient_boosting, X_test, y_test))

results_df = pd.DataFrame(results)
print(results_df.round(4))

                 model  decision_threshold  accuracy  precision  recall  \
0                Dummy                 0.5    0.9670     0.0000  0.0000   
1  Logistic Regression                 0.5    0.8475     0.1595  0.8485   
2        Random Forest                 0.5    0.9900     0.8594  0.8333   
3    Gradient Boosting                 0.5    0.9895     0.8814  0.7879   

       f1  roc_auc  pr_auc  
0  0.0000   0.5000  0.0330  
1  0.2686   0.9202  0.4192  
2  0.8462   0.9910  0.8930  
3  0.8320   0.9943  0.9145  


## False-negative analysis

Identify failures that were not detected by the models.

## Observations from model comparison

- Random Forest provides the best precision/recall balance.
- Gradient Boosting achieves the best ROC-AUC and PR-AUC.
- Accuracy alone is insufficient for this imbalanced target.
- Further changes should begin with cross-validation.

### Reconstruct the unscaled test set

Models were trained on scaled features, but error analysis requires the original units, such as rpm, Nm, W and min.

We therefore recreate the split using the same parameters as in data preparation:

- `test_size=0.2`,
- `random_state=42`,
- `stratify=y_raw`.

This preserves the same test records while restoring physically interpretable values.

In [12]:
from sklearn.model_selection import train_test_split

df_clean = pd.read_csv('../data/processed/produkcja_clean.csv')

X_raw = df_clean.drop(columns=['Machine failure'])
y_raw = df_clean['Machine failure']

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw,
    y_raw,
    test_size=0.2,
    random_state=42,
    stratify=y_raw
)

X_test_raw = X_test_raw.copy()

In [13]:
print("X_test scaled shape:", X_test.shape)
print("X_test raw shape:", X_test_raw.shape)

print("y_test shape:", y_test.shape)
print("y_test_raw shape:", y_test_raw.shape)

X_test scaled shape: (2000, 8)
X_test raw shape: (2000, 8)
y_test shape: (2000,)
y_test_raw shape: (2000,)


In [14]:
print((y_test.reset_index(drop=True) == y_test_raw.reset_index(drop=True)).all())

True


In [15]:
debug_df = X_test_raw.copy()
debug_df['y_true'] = y_test_raw.values

models = {
    'log_reg': log_reg,
    'random_forest': random_forest,
    'gradient_boosting': gradient_boosting
}

for name, model in models.items():
    proba = model.predict_proba(X_test)[:, 1]
    debug_df[f"{name}_proba"] = proba
    debug_df[f"{name}_pred"] = (proba > DECISION_THRESHOLD).astype(int)
    

print(debug_df.head())
print(debug_df.shape)

      Type_L  Type_M  Process temperature [K]  Temperature difference  \
7566       1       0                    311.0                    10.7   
8199       1       0                    310.7                    11.5   
9316       0       0                    309.4                    10.9   
2882       1       0                    309.6                     9.0   
4293       1       0                    310.0                     8.2   

      Rotational speed [rpm]  Torque [Nm]  Power [W]  Tool wear [min]  y_true  \
7566                    1613         34.7    5861.28              142       0   
8199                    1737         27.0    4911.25              225       1   
9316                    1417         41.2    6113.58              155       0   
2882                    1684         32.8    5784.22               53       0   
4293                    1393         44.9    6549.77              208       0   

      log_reg_proba  log_reg_pred  random_forest_proba  random_forest_pred

In [16]:
debug_df[[
    "y_true",
    "log_reg_pred",
    "log_reg_proba",
    "random_forest_pred",
    "random_forest_proba",
    "gradient_boosting_pred",
    "gradient_boosting_proba"
]].head(10)

,y_true,log_reg_pred,log_reg_proba,random_forest_pred,random_forest_proba,gradient_boosting_pred,gradient_boosting_proba
7566,0,0,0.041565,0,0.000,0,0.000622
8199,1,0,0.070138,0,0.150,0,0.138585
9316,0,0,0.086594,0,0.005,0,0.000668
2882,0,0,0.032116,0,0.000,0,0.000638
4293,0,1,0.879893,0,0.410,0,0.037122
4261,0,0,0.183033,0,0.000,0,0.002261
4485,0,0,0.090754,0,0.000,0,0.002235
2518,0,0,0.115763,0,0.000,0,0.000668
8084,0,0,0.095162,0,0.000,0,0.000622
9661,0,0,0.306636,0,0.085,0,0.054976


In [17]:
get_false_negatives('log_reg')

,Type_L,Type_M,Process temperature [K],Temperature difference,Rotational speed [rpm],Torque [Nm],Power [W],Tool wear [min],y_true,log_reg_proba,log_reg_pred,random_forest_proba,random_forest_pred,gradient_boosting_proba,gradient_boosting_pred
8199,1,0,310.7,11.5,1737,27.0,4911.25,225,1,0.070138,0,0.150,0,0.138585,0
4151,0,1,310.4,8.5,1373,48.0,6901.45,73,1,0.443058,0,0.895,1,0.943080,1
4475,1,0,310.5,7.8,1351,41.8,5913.71,10,1,0.163843,0,0.415,0,0.965152,1
4034,1,0,310.8,8.8,1615,29.0,4904.55,235,1,0.431925,0,0.150,0,0.053293,0
4833,1,0,311.9,8.5,1377,41.6,5998.68,34,1,0.131180,0,0.340,0,0.965545,1
8609,1,0,308.3,10.9,1475,40.5,6255.70,222,1,0.249070,0,0.110,0,0.057842,0
9018,1,0,308.1,10.8,1615,35.4,5986.93,217,1,0.132736,0,0.035,0,0.053958,0
442,1,0,308.5,11.1,1399,61.5,9009.93,61,1,0.399294,0,0.710,1,0.975880,1
3787,1,0,310.8,8.5,1377,47.3,6820.62,22,1,0.248337,0,0.785,1,0.943080,1
7510,1,0,311.8,11.3,1524,38.9,6208.16,214,1,0.137707,0,0.125,0,0.045325,0


In [18]:
print(get_false_negatives('random_forest'))

      Type_L  Type_M  Process temperature [K]  Temperature difference  \
8199       1       0                    310.7                    11.5   
4475       1       0                    310.5                     7.8   
4034       1       0                    310.8                     8.8   
4833       1       0                    311.9                     8.5   
2941       0       1                    309.6                     8.9   
8609       1       0                    308.3                    10.9   
8026       1       0                    311.9                    11.2   
9663       1       0                    310.1                    11.0   
9758       1       0                    309.8                    11.2   
9018       1       0                    308.1                    10.8   
7510       1       0                    311.8                    11.3   

      Rotational speed [rpm]  Torque [Nm]  Power [W]  Tool wear [min]  y_true  \
8199                    1737         27.0 

In [19]:
print(get_false_negatives('gradient_boosting'))

      Type_L  Type_M  Process temperature [K]  Temperature difference  \
8199       1       0                    310.7                    11.5   
4034       1       0                    310.8                     8.8   
2494       1       0                    308.8                     9.7   
2941       0       1                    309.6                     8.9   
1085       1       0                    307.8                    10.8   
8609       1       0                    308.3                    10.9   
8026       1       0                    311.9                    11.2   
9653       1       0                    309.9                    10.9   
3760       1       0                    310.9                     8.6   
9663       1       0                    310.1                    11.0   
9758       1       0                    309.8                    11.2   
3829       0       0                    310.9                     8.6   
9018       1       0                    308.1      

In [20]:
fn_summary = pd.DataFrame({
    'model': ['log_reg', 'random_forest', 'gradient_boosting'],
    'false_negatives': [get_false_negatives('log_reg').shape[0],
                        get_false_negatives('random_forest').shape[0],
                        get_false_negatives('gradient_boosting').shape[0]]
})
print(fn_summary)

               model  false_negatives
0            log_reg               10
1      random_forest               11
2  gradient_boosting               14


In [21]:
fn_log_index = set(get_false_negatives('log_reg').index)
fn_rf_index = set(get_false_negatives('random_forest').index)
fn_gb_index = set(get_false_negatives('gradient_boosting').index)

print("Logistic Regression FN:", len(fn_log_index))
print("Random Forest FN:", len(fn_rf_index))
print("Gradient Boosting FN:", len(fn_gb_index))

Logistic Regression FN: 10
Random Forest FN: 11
Gradient Boosting FN: 14


In [22]:
fn_all_index = fn_log_index & fn_rf_index & fn_gb_index

print("Awarie przeoczone przez wszystkie modele:", len(fn_all_index))
print(fn_all_index)

Awarie przeoczone przez wszystkie modele: 5
{8609, 4034, 8199, 7510, 9018}


In [23]:
fn_all = debug_df.loc[list(fn_all_index)].copy()
print(fn_all)

      Type_L  Type_M  Process temperature [K]  Temperature difference  \
8609       1       0                    308.3                    10.9   
4034       1       0                    310.8                     8.8   
8199       1       0                    310.7                    11.5   
7510       1       0                    311.8                    11.3   
9018       1       0                    308.1                    10.8   

      Rotational speed [rpm]  Torque [Nm]  Power [W]  Tool wear [min]  y_true  \
8609                    1475         40.5    6255.70              222       1   
4034                    1615         29.0    4904.55              235       1   
8199                    1737         27.0    4911.25              225       1   
7510                    1524         38.9    6208.16              214       1   
9018                    1615         35.4    5986.93              217       1   

      log_reg_proba  log_reg_pred  random_forest_proba  random_forest_pred

In [24]:
overlap_summary = pd.DataFrame({
    "comparison": [
        "Logistic Regression ∩ Random Forest",
        "Logistic Regression ∩ Gradient Boosting",
        "Random Forest ∩ Gradient Boosting",
        "All three models"
    ],
    "shared_false_negatives": [
        len(fn_log_index & fn_rf_index),
        len(fn_log_index & fn_gb_index),
        len(fn_rf_index & fn_gb_index),
        len(fn_all_index)
    ]
})

print(overlap_summary)

                                comparison  shared_false_negatives
0      Logistic Regression ∩ Random Forest                       7
1  Logistic Regression ∩ Gradient Boosting                       5
2        Random Forest ∩ Gradient Boosting                       9
3                         All three models                       5


In [25]:
debug_df["missed_by_log_reg"] = (
    (debug_df["y_true"] == 1) &
    (debug_df["log_reg_pred"] == 0)
)

debug_df["missed_by_random_forest"] = (
    (debug_df["y_true"] == 1) &
    (debug_df["random_forest_pred"] == 0)
)

debug_df["missed_by_gradient_boosting"] = (
    (debug_df["y_true"] == 1) &
    (debug_df["gradient_boosting_pred"] == 0)
)

debug_df["missed_by_n_models"] = (
    debug_df["missed_by_log_reg"].astype(int) +
    debug_df["missed_by_random_forest"].astype(int) +
    debug_df["missed_by_gradient_boosting"].astype(int)
)

In [26]:
missed_count_summary = (
    debug_df[debug_df["y_true"] == 1]["missed_by_n_models"]
    .value_counts()
    .sort_index()
)

print(missed_count_summary)

missed_by_n_models
0    47
1     8
2     6
3     5
Name: count, dtype: int64


In [27]:
hard_cases = debug_df[
    (debug_df["y_true"] == 1) &
    (debug_df["missed_by_n_models"] > 0)
].copy()
diagnostic_cols = [
    "y_true",
    "log_reg_pred",
    "log_reg_proba",
    "random_forest_pred",
    "random_forest_proba",
    "gradient_boosting_pred",
    "gradient_boosting_proba",
    "Process temperature [K]",
    "Temperature difference",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Power [W]",
    "Tool wear [min]",
    "Type_L",
    "Type_M"
]

print(
    hard_cases[diagnostic_cols + ["missed_by_n_models"]]
    .sort_values("missed_by_n_models", ascending=False)
)

      y_true  log_reg_pred  log_reg_proba  random_forest_pred  \
8199       1             0       0.070138                   0   
8609       1             0       0.249070                   0   
4034       1             0       0.431925                   0   
9018       1             0       0.132736                   0   
7510       1             0       0.137707                   0   
9758       1             1       0.778214                   0   
4475       1             0       0.163843                   0   
8026       1             1       0.635897                   0   
9663       1             1       0.590242                   0   
2941       1             1       0.605943                   0   
4833       1             0       0.131180                   0   
4151       1             0       0.443058                   1   
9653       1             1       0.826901                   1   
2494       1             1       0.934708                   1   
1085       1             

### Summarize missed failures

Count false negatives and compare model overlap.

### Inspect common false negatives

Review failures missed by all baseline models.

In [28]:
df_raw = pd.read_csv('../data/raw/produkcja.csv')
df_raw['Power [W]'] = df_raw['Rotational speed [rpm]'] * df_raw['Torque [Nm]'] * (2 * np.pi / 60)
df_raw['Temperature difference'] = df_raw['Process temperature [K]'] - df_raw['Air temperature [K]']
df_raw['Kryterium_OSF'] = df_raw['Tool wear [min]'] * df_raw['Torque [Nm]']

In [29]:
fn_every = sorted(fn_all_index)

print(f"Common false negatives: {len(fn_every)}")
print(fn_every)

Common false negatives: 5
[4034, 7510, 8199, 8609, 9018]


In [30]:
suspect_cases = df_raw.loc[fn_every].copy()
print(suspect_cases)

       UDI Product ID Type  Air temperature [K]  Process temperature [K]  \
4034  4035     L51214    L                302.0                    310.8   
7510  7511     L54690    L                300.5                    311.8   
8199  8200     L55379    L                299.2                    310.7   
8609  8610     L55789    L                297.4                    308.3   
9018  9019     L56198    L                297.3                    308.1   

      Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  Machine failure  \
4034                    1615         29.0              235                1   
7510                    1524         38.9              214                1   
8199                    1737         27.0              225                1   
8609                    1475         40.5              222                1   
9018                    1615         35.4              217                1   

      TWF  HDF  PWF  OSF  RNF    Power [W]  Temperature difference  

### Interpret false-negative patterns

Group missed failures by suspected mechanisms and boundary cases.

442 - very close to the lower PWF boundary

3760, 3787, 3829, 4151, 4833, 4475 - close to the HDF boundaries for rotational speed and temperature difference

1085, 2494, 9653, 9663, 8026 - the models did not include the OSF criterion

2941, 9758 - high rotational speed combined with high tool wear [TWF]; EDA did not show a clear relationship between rotational speed and TWF

7510, 9018, 4034, 8609, 8199 - high tool wear [TWF]


3787, 442, 3760, 3829, 1085, 2494, 9653, 4151 - missed by one model

4833, 2941, 9663, 8026, 4475, 9758 - missed by two models

7510, 9018, 4034, 8609, 8199 - missed by all three models

In [31]:
print('debug_df:', debug_df.shape)
print("Random Forest:", type(random_forest).__name__)
print("Gradient Boosting:", type(gradient_boosting).__name__)

debug_df: (2000, 19)
Random Forest: RandomForestClassifier
Gradient Boosting: GradientBoostingClassifier


## False-positive analysis

Inspect normal records incorrectly flagged as failures.

In [32]:
fp_rf = debug_df[(debug_df['y_true']==0) & (debug_df['random_forest_pred']== 1)].copy()

fp_gb = debug_df[(debug_df['y_true']==0) & (debug_df['gradient_boosting_pred']== 1)].copy()

print("False positives - Random Forest:", len(fp_rf))
print("False positives - Gradient Boosting:", len(fp_gb))

False positives - Random Forest: 9
False positives - Gradient Boosting: 7


In [33]:
print(fp_rf)
print(fp_gb)


      Type_L  Type_M  Process temperature [K]  Temperature difference  \
3764       1       0                    311.0                     8.6   
4898       0       0                    312.4                     8.8   
8198       0       1                    310.7                    11.4   
7081       1       0                    310.4                     9.7   
7677       1       0                    311.7                    11.1   
4115       0       0                    310.6                     8.6   
4110       1       0                    310.6                     8.6   
7255       0       1                    310.3                    10.1   
4862       1       0                    312.1                     8.6   

      Rotational speed [rpm]  Torque [Nm]  Power [W]  Tool wear [min]  y_true  \
3764                    1329         54.6    7598.82              176       0   
4898                    1460         53.7    8210.24              214       0   
8198                    13

In [34]:
print("RF — wartości y_true:")
print(fp_rf["y_true"].value_counts())

print("\nRF — wartości predykcji:")
print(fp_rf["random_forest_pred"].value_counts())

print("\nGB — wartości y_true:")
print(fp_gb["y_true"].value_counts())

print("\nGB — wartości predykcji:")
print(fp_gb["gradient_boosting_pred"].value_counts())

RF — wartości y_true:
y_true
0    9
Name: count, dtype: int64

RF — wartości predykcji:
random_forest_pred
1    9
Name: count, dtype: int64

GB — wartości y_true:
y_true
0    7
Name: count, dtype: int64

GB — wartości predykcji:
gradient_boosting_pred
1    7
Name: count, dtype: int64


In [35]:
fp_rf_summary = fp_rf.drop(columns=["log_reg_pred", "log_reg_proba", "gradient_boosting_pred", "gradient_boosting_proba", "missed_by_log_reg", "missed_by_gradient_boosting", "missed_by_n_models", "missed_by_random_forest"]).copy()
print(fp_rf_summary)

      Type_L  Type_M  Process temperature [K]  Temperature difference  \
3764       1       0                    311.0                     8.6   
4898       0       0                    312.4                     8.8   
8198       0       1                    310.7                    11.4   
7081       1       0                    310.4                     9.7   
7677       1       0                    311.7                    11.1   
4115       0       0                    310.6                     8.6   
4110       1       0                    310.6                     8.6   
7255       0       1                    310.3                    10.1   
4862       1       0                    312.1                     8.6   

      Rotational speed [rpm]  Torque [Nm]  Power [W]  Tool wear [min]  y_true  \
3764                    1329         54.6    7598.82              176       0   
4898                    1460         53.7    8210.24              214       0   
8198                    13

### Random Forest false positives

List and inspect normal records flagged by Random Forest.

False positives dla Random Forest: 3764
4898
8198
7081
7677
4115
4110
7255
4862

In [36]:
print(fp_rf_summary["random_forest_proba"].min())
print(fp_rf_summary["random_forest_proba"].max())

0.53
0.75


### Random Forest probability range

Check how close the false positives are to the decision threshold.

Random Forest probability range: [0.53, 0.75]

In [37]:
fp_gb_summary = fp_gb.drop(columns=["log_reg_pred", "log_reg_proba", "random_forest_pred", "random_forest_proba", "missed_by_log_reg", "missed_by_gradient_boosting", "missed_by_n_models", "missed_by_random_forest"]).copy()
print(fp_gb_summary)

      Type_L  Type_M  Process temperature [K]  Temperature difference  \
6925       0       1                    311.6                    10.5   
3764       1       0                    311.0                     8.6   
7677       1       0                    311.7                    11.1   
1507       0       1                    308.6                    10.6   
7935       0       1                    311.9                    11.1   
4862       1       0                    312.1                     8.6   
9671       1       0                    310.2                    11.2   

      Rotational speed [rpm]  Torque [Nm]  Power [W]  Tool wear [min]  y_true  \
6925                    1266         55.5    7357.92              210       0   
3764                    1329         54.6    7598.82              176       0   
7677                    1428         52.6    7865.79              208       0   
1507                    1376         55.7    8026.06              214       0   
7935      

### Gradient Boosting false positives

List and inspect normal records flagged by Gradient Boosting.

False positives dla Gradient Boosting: 6925 3764 7677 1507 7935 4862 9671

In [38]:
print(fp_gb_summary["gradient_boosting_proba"].min().round(2))
print(fp_gb_summary["gradient_boosting_proba"].max().round(2))

0.54
0.85


### Gradient Boosting probability range

Check the confidence range of false-positive predictions.

Gradient Boosting probability range: (0.54, 0.85)

In [39]:
common_fp = fp_rf_summary.merge(fp_gb_summary, left_index=True, right_index=True, suffixes=('_rf', '_gb'))
print(common_fp)

      Type_L_rf  Type_M_rf  Process temperature [K]_rf  \
3764          1          0                       311.0   
7677          1          0                       311.7   
4862          1          0                       312.1   

      Temperature difference_rf  Rotational speed [rpm]_rf  Torque [Nm]_rf  \
3764                        8.6                       1329            54.6   
7677                       11.1                       1428            52.6   
4862                        8.6                       1326            55.7   

      Power [W]_rf  Tool wear [min]_rf  y_true_rf  random_forest_proba  ...  \
3764       7598.82                 176          0                0.670  ...   
7677       7865.79                 208          0                0.645  ...   
4862       7734.41                 114          0                0.665  ...   

      Type_M_gb  Process temperature [K]_gb  Temperature difference_gb  \
3764          0                       311.0                    

In [40]:
common_fp[['random_forest_proba', 'gradient_boosting_proba']].round(3)

,random_forest_proba,gradient_boosting_proba
3764,0.670,0.543
7677,0.645,0.855
4862,0.665,0.548


In [41]:

fp_rf_summary['random_forest_margin'] = (fp_rf_summary['random_forest_proba'] - DECISION_THRESHOLD)

fp_gb_summary['gradient_boosting_margin'] = (fp_gb_summary['gradient_boosting_proba'] - DECISION_THRESHOLD)

In [42]:
print(fp_rf_summary[['random_forest_proba', 'random_forest_margin']].round(3))

      random_forest_proba  random_forest_margin
3764                0.670                 0.170
4898                0.610                 0.110
8198                0.560                 0.060
7081                0.640                 0.140
7677                0.645                 0.145
4115                0.580                 0.080
4110                0.750                 0.250
7255                0.530                 0.030
4862                0.665                 0.165


In [43]:
print(fp_gb_summary[['gradient_boosting_proba', 'gradient_boosting_margin']].round(3))

      gradient_boosting_proba  gradient_boosting_margin
6925                    0.742                     0.242
3764                    0.543                     0.043
7677                    0.855                     0.355
1507                    0.850                     0.350
7935                    0.589                     0.089
4862                    0.548                     0.048
9671                    0.657                     0.157


In [44]:
osf_threshold = 11003.2

fp_rf_summary['OSF_distance'] = (fp_rf_summary['Torque [Nm]'] * fp_rf_summary['Tool wear [min]']) - osf_threshold
fp_gb_summary['OSF_distance'] = (fp_gb_summary['Torque [Nm]'] * fp_gb_summary['Tool wear [min]']) - osf_threshold

fp_rf_summary['OSF_like'] = fp_rf_summary['OSF_distance'] >= 0
fp_gb_summary['OSF_like'] = fp_gb_summary['OSF_distance'] >= 0


In [45]:
fp_rf_summary['HDF_like'] = fp_rf_summary['Temperature difference'].between(7.6, 8.6) & fp_rf_summary['Rotational speed [rpm]'].between(1212, 1379)
fp_gb_summary['HDF_like'] = fp_gb_summary['Temperature difference'].between(7.6, 8.6) & fp_gb_summary['Rotational speed [rpm]'].between(1212, 1379)


In [46]:
fp_rf_summary['PWF_like'] = fp_rf_summary['PWF_like'] = (
    ((fp_rf_summary['Power [W]'] > 1148) & (fp_rf_summary['Power [W]'] < 3477))
    |
    ((fp_rf_summary['Power [W]'] > 9004) & (fp_rf_summary['Power [W]'] < 10469))
)
fp_gb_summary['PWF_like'] = fp_gb_summary['PWF_like'] = (
    ((fp_gb_summary['Power [W]'] > 1148) & (fp_gb_summary['Power [W]'] < 3477))
    |
    ((fp_gb_summary['Power [W]'] > 9004) & (fp_gb_summary['Power [W]'] < 10469))
)


In [47]:
fp_rf_summary['TWF_like'] = fp_rf_summary['Tool wear [min]'] > 198
fp_gb_summary['TWF_like'] = fp_gb_summary['Tool wear [min]'] > 198

In [48]:
print(fp_rf_summary.head(15))

      Type_L  Type_M  Process temperature [K]  Temperature difference  \
3764       1       0                    311.0                     8.6   
4898       0       0                    312.4                     8.8   
8198       0       1                    310.7                    11.4   
7081       1       0                    310.4                     9.7   
7677       1       0                    311.7                    11.1   
4115       0       0                    310.6                     8.6   
4110       1       0                    310.6                     8.6   
7255       0       1                    310.3                    10.1   
4862       1       0                    312.1                     8.6   

      Rotational speed [rpm]  Torque [Nm]  Power [W]  Tool wear [min]  y_true  \
3764                    1329         54.6    7598.82              176       0   
4898                    1460         53.7    8210.24              214       0   
8198                    13

In [49]:
print(fp_gb_summary.head(15))

      Type_L  Type_M  Process temperature [K]  Temperature difference  \
6925       0       1                    311.6                    10.5   
3764       1       0                    311.0                     8.6   
7677       1       0                    311.7                    11.1   
1507       0       1                    308.6                    10.6   
7935       0       1                    311.9                    11.1   
4862       1       0                    312.1                     8.6   
9671       1       0                    310.2                    11.2   

      Rotational speed [rpm]  Torque [Nm]  Power [W]  Tool wear [min]  y_true  \
6925                    1266         55.5    7357.92              210       0   
3764                    1329         54.6    7598.82              176       0   
7677                    1428         52.6    7865.79              208       0   
1507                    1376         55.7    8026.06              214       0   
7935      

In [50]:
print(f"RF OSF: {fp_rf_summary['OSF_like'].sum()}")
print(f"RF HDF: {fp_rf_summary['HDF_like'].sum()}")
print(f"RF PWF: {fp_rf_summary['PWF_like'].sum()}")
print(f"RF TWF: {fp_rf_summary['TWF_like'].sum()}")
print(f"GB OSF: {fp_gb_summary['OSF_like'].sum()}")
print(f"GB HDF: {fp_gb_summary['HDF_like'].sum()}")
print(f"GB PWF: {fp_gb_summary['PWF_like'].sum()}")
print(f"GB TWF: {fp_gb_summary['TWF_like'].sum()}")

RF OSF: 1
RF HDF: 4
RF PWF: 0
RF TWF: 5
GB OSF: 2
GB HDF: 2
GB PWF: 0
GB TWF: 5


In [51]:
cols_like = ['OSF_like', 'HDF_like', 'PWF_like', 'TWF_like']

fp_rf_summary[cols_like].sum(axis=1)

3764    1
4898    2
8198    1
7081    1
7677    1
4115    1
4110    1
7255    1
4862    1
dtype: int64

In [52]:
fp_gb_summary[cols_like].sum(axis=1)

6925    2
3764    1
7677    1
1507    2
7935    1
4862    1
9671    1
dtype: int64

## Export baseline results

Save metrics and error-analysis tables.

In [53]:
from pathlib import Path

# 1. Utworzenie folderu na wyniki
results_path = Path("../results")
results_path.mkdir(parents=True, exist_ok=True)

# 2. Zapis metryk modeli
results_df.round(6).to_csv(
    results_path / "model_results_baseline.csv",
    index=False
)

# 3. Przygotowanie i zapis false negatives
false_negatives_export = (
    hard_cases[
        diagnostic_cols + ["missed_by_n_models"]
    ]
    .sort_values(
        by="missed_by_n_models",
        ascending=False
    )
    .copy()
)
proba_columns = [
    "log_reg_proba",
    "random_forest_proba",
    "gradient_boosting_proba"
]

false_negatives_export[proba_columns] = (
    false_negatives_export[proba_columns].round(3)
)
false_negatives_export.to_csv(
    results_path / "false_negatives_baseline.csv",
    index=True,
    index_label="record_index"
)

# 4. Przygotowanie false positives
fp_rf_export = fp_rf_summary.rename(
    columns={"Kryterium_OSF": "OSF_distance"}
).copy()

fp_gb_export = fp_gb_summary.rename(
    columns={"Kryterium_OSF": "OSF_distance"}
).copy()
for df in [fp_rf_export, fp_gb_export]:

    df["OSF_distance"] = (
        df["Tool wear [min]"] * df["Torque [Nm]"] - 11003.2
    )

    df["OSF_like"] = df["OSF_distance"] >= 0

    df["HDF_like"] = (
        df["Temperature difference"].between(7.6, 8.6)
        & df["Rotational speed [rpm]"].between(1212, 1379)
    )

    df["PWF_like"] = (
        df["Power [W]"].between(1148.44, 3477.24)
        | df["Power [W]"].between(9004.43, 10469.92)
    )

    df["TWF_like"] = df["Tool wear [min]"].between(198, 253)
like_columns = [
    "OSF_like",
    "HDF_like",
    "PWF_like",
    "TWF_like"
]

fp_rf_export["n_like_mechanisms"] = (
    fp_rf_export[like_columns].sum(axis=1)
)

fp_gb_export["n_like_mechanisms"] = (
    fp_gb_export[like_columns].sum(axis=1)
)

fp_rf_export["model"] = "Random Forest"
fp_gb_export["model"] = "Gradient Boosting"

fp_rf_export = fp_rf_export.rename(
    columns={
        "random_forest_proba": "predicted_probability",
        "random_forest_margin": "margin"
    }
)

fp_gb_export = fp_gb_export.rename(
    columns={
        "gradient_boosting_proba": "predicted_probability",
        "gradient_boosting_margin": "margin"
    }
)

false_positive_columns = [
    "model",
    "predicted_probability",
    "margin",
    "Process temperature [K]",
    "Temperature difference",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Power [W]",
    "Tool wear [min]",
    "OSF_distance",
    "OSF_like",
    "HDF_like",
    "PWF_like",
    "TWF_like",
    "n_like_mechanisms"
]

false_positives_export = pd.concat(
    [
        fp_rf_export[false_positive_columns],
        fp_gb_export[false_positive_columns]
    ],
    axis=0
)

false_positives_export = (
    false_positives_export
    .reset_index()
    .rename(columns={"index": "record_index"})
    .sort_values(
        by=["model", "predicted_probability"],
        ascending=[True, False]
    )
)
columns_to_round = [
    "predicted_probability",
    "margin",
    "OSF_distance"
]

false_positives_export[columns_to_round] = (
    false_positives_export[columns_to_round].round(3)
)
false_positives_export.to_csv(
    results_path / "false_positives_baseline.csv",
    index=False
)

print("Zapisano pliki:")
print(results_path / "model_results_baseline.csv")
print(results_path / "false_negatives_baseline.csv")
print(results_path / "false_positives_baseline.csv")

Zapisano pliki:
..\results\model_results_baseline.csv
..\results\false_negatives_baseline.csv
..\results\false_positives_baseline.csv
